<a href="https://colab.research.google.com/github/swapnasaina274/malicious-mobile-app-detection/blob/main/Apkextractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q androguard pandas


In [2]:
from google.colab import files

uploaded = files.upload()

apk_path = next(
    name for name in uploaded.keys()
    if name.lower().endswith(".apk")
)

print("APK selected:", apk_path)


Saving com.iris.gallery_16.apk to com.iris.gallery_16.apk
APK selected: com.iris.gallery_16.apk


In [4]:
import os

print("APK path:", apk_path)
print("APK size:", round(os.path.getsize(apk_path) / (1024 * 1024), 2), "MB")

APK path: com.iris.gallery_16.apk
APK size: 7.93 MB


In [6]:
import os
import zipfile

print("File exists:", os.path.exists(apk_path))
print("File size (MB):", round(os.path.getsize(apk_path) / 1024**2, 2))

with zipfile.ZipFile(apk_path, "r") as z:
    print("ZIP integrity:", z.testzip())
    print("Manifest exists:", "AndroidManifest.xml" in z.namelist())
    print("DEX files:", [n for n in z.namelist()
                         if n.endswith(".dex")])

File exists: True
File size (MB): 7.93
ZIP integrity: None
Manifest exists: True
DEX files: ['classes.dex']


In [7]:
!apt-get update -qq && apt-get install -y -qq apktool

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package android-framework-res.
(Reading database ... 122806 files and directories currently installed.)
Preparing to unpack .../00-android-framework-res_1%3a14~beta1-2build3_all.deb ...
Unpacking android-framework-res (1:14~beta1-2build3) ...
Selecting previously unselected package android-liblog:amd64.
Preparing to unpack .../01-android-liblog_1%3a34.0.4-1build3_amd64.deb ...
Unpacking android-liblog:amd64 (1:34.0.4-1build3) ...
Selecting previously unselected package android-libbase:amd64.
Preparing to unpack .../02-android-libbase_1%3a34.0.4-1build3_amd64.deb ...
Unpacking android-libbase:amd64 (1:34.0.4-1build3) ...
Selecting previously unselected package android-libbacktrace:amd64.
Preparing to unpack .../03-android-libback

In [8]:
import subprocess
import os
import time

output_dir = "/content/apk_decoded"
start = time.time()

result = subprocess.run(
    ["apktool", "d", "-f", "-s", apk_path, "-o", output_dir],
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", result.returncode)
print("Time:", round(time.time() - start, 2), "seconds")
print(result.stderr[-2000:])

manifest_path = os.path.join(output_dir, "AndroidManifest.xml")
print("Manifest decoded:", os.path.exists(manifest_path))

if os.path.exists(manifest_path):
    with open(manifest_path, encoding="utf-8") as f:
        print(f.read()[:5000])

Return code: 0
Time: 6.05 seconds

Manifest decoded: True
<?xml version="1.0" encoding="utf-8" standalone="no"?><manifest xmlns:android="http://schemas.android.com/apk/res/android" android:compileSdkVersion="35" android:compileSdkVersionCodename="15" package="com.iris.gallery" platformBuildVersionCode="35" platformBuildVersionName="15">
    <uses-permission android:name="android.permission.READ_MEDIA_IMAGES"/>
    <uses-permission android:name="android.permission.READ_MEDIA_VIDEO"/>
    <uses-permission android:name="android.permission.READ_MEDIA_VISUAL_USER_SELECTED"/>
    <uses-permission android:name="android.permission.ACCESS_MEDIA_LOCATION"/>
    <uses-permission android:maxSdkVersion="32" android:name="android.permission.READ_EXTERNAL_STORAGE"/>
    <uses-permission android:maxSdkVersion="29" android:name="android.permission.WRITE_EXTERNAL_STORAGE"/>
    <uses-permission android:name="android.permission.MANAGE_EXTERNAL_STORAGE"/>
    <uses-permission android:name="android.permiss

In [9]:

import xml.etree.ElementTree as ET
import pandas as pd
import json
import os
import time

ANDROID_NS = "{http://schemas.android.com/apk/res/android}"

def android_attr(element, name, default=""):
    return element.get(ANDROID_NS + name, default)

def extract_manifest_features(manifest_path):
    start = time.time()

    root = ET.parse(manifest_path).getroot()
    app = root.find("application")

    permissions = sorted({
        android_attr(node, "name")
        for node in root.findall("uses-permission")
        if android_attr(node, "name")
    })

    components = {}
    for tag in ["activity", "activity-alias", "service",
                "receiver", "provider"]:
        components[tag] = [
            android_attr(node, "name")
            for node in (app.findall(tag) if app is not None else [])
            if android_attr(node, "name")
        ]

    features = {
        "package_name": root.get("package", ""),
        "min_sdk": "",
        "target_sdk": "",
        "permission_count": len(permissions),
        "activity_count": len(components["activity"]),
        "activity_alias_count": len(components["activity-alias"]),
        "service_count": len(components["service"]),
        "receiver_count": len(components["receiver"]),
        "provider_count": len(components["provider"]),
        "exported_component_count": 0,
        "intent_filter_count": 0,
        "application_allow_backup": "",
        "application_debuggable": "",
        "permissions": json.dumps(permissions),
    }

    sdk = root.find("uses-sdk")
    if sdk is not None:
        features["min_sdk"] = android_attr(sdk, "minSdkVersion")
        features["target_sdk"] = android_attr(sdk, "targetSdkVersion")

    if app is not None:
        features["application_allow_backup"] = android_attr(
            app, "allowBackup"
        )
        features["application_debuggable"] = android_attr(
            app, "debuggable"
        )

        for component_tag in [
            "activity", "activity-alias", "service",
            "receiver", "provider"
        ]:
            for component in app.findall(component_tag):
                if android_attr(component, "exported").lower() == "true":
                    features["exported_component_count"] += 1

                features["intent_filter_count"] += len(
                    component.findall("intent-filter")
                )

    # One binary feature per permission
    for permission in permissions:
        features["perm_" + permission] = 1

    # Store component names separately for reporting
    features["components"] = json.dumps(components)

    print(f"Extraction completed in {time.time() - start:.2f} seconds")
    return features


manifest_path = "/content/apk_decoded/AndroidManifest.xml"

features = extract_manifest_features(manifest_path)

os.makedirs("/content/apk_output", exist_ok=True)

pd.DataFrame([features]).to_csv(
    "/content/apk_output/apk_features.csv", index=False
)

with open("/content/apk_output/apk_report.json", "w") as f:
    json.dump(features, f, indent=2)

print("Package:", features["package_name"])
print("Permissions:", features["permission_count"])
print("Activities:", features["activity_count"])
print("Services:", features["service_count"])
print("Receivers:", features["receiver_count"])
print("Providers:", features["provider_count"])
print("Total feature columns:", len(features))

display(pd.DataFrame([features]).T)


Extraction completed in 0.00 seconds
Package: com.iris.gallery
Permissions: 14
Activities: 1
Services: 0
Receivers: 4
Providers: 2
Total feature columns: 29


,0
package_name,com.iris.gallery
min_sdk,
target_sdk,
permission_count,14
activity_count,1
activity_alias_count,0
service_count,0
receiver_count,4
provider_count,2
exported_component_count,3


In [10]:
from google.colab import files

files.download("/content/apk_output/apk_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import os

for root, dirs, filenames in os.walk("/content"):
    dirs[:] = [
        d for d in dirs
        if d not in {"node_modules", ".git", "__pycache__"}
    ]

    for name in filenames:
        if name.endswith((".pkl", ".joblib", ".csv", ".ipynb")):
            print(os.path.join(root, name))

/content/apk_features.csv
/content/apk_output/apk_features.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv


In [19]:

import os
import zipfile

model_path = "/content/drive/MyDrive/Malicious project work/champion_lightgbm_model.pkl"

print("File exists:", os.path.isfile(model_path))
print("File size:", os.path.getsize(model_path), "bytes")

with open(model_path, "rb") as f:
    header = f.read(32)

print("First 32 bytes:", repr(header))
print("ZIP archive:", zipfile.is_zipfile(model_path))


File exists: True
File size: 1765468 bytes
First 32 bytes: b'\x80\x04\x95\xdf\x01\x00\x00\x00\x00\x00\x00\x8c\x10lightgbm.sklearn\x94\x8c\x0e'
ZIP archive: False


In [20]:
%pip install -q joblib lightgbm

In [21]:
import joblib
import os

model_path = (
    "/content/drive/MyDrive/Malicious project work/"
    "champion_lightgbm_model.pkl"
)

champion_model = joblib.load(model_path)

print("Champion model loaded!")
print("Model class:", type(champion_model))
print("Expected features:", champion_model.n_features_in_)

Champion model loaded!
Model class: <class 'lightgbm.sklearn.LGBMClassifier'>
Expected features: 470


In [22]:
if hasattr(champion_model, "feature_name_"):
    print(champion_model.feature_name_[:30])

elif hasattr(champion_model, "feature_names_in_"):
    print(champion_model.feature_names_in_[:30])

elif hasattr(champion_model, "booster_"):
    print(champion_model.booster_.feature_name()[:30])

['ACCESS_PERSONAL_INFO___', 'ALTER_PHONE_STATE___', 'ANTI_DEBUG_____', 'CREATE_FOLDER_____', 'CREATE_PROCESS`_____', 'CREATE_THREAD_____', 'DEVICE_ACCESS_____', 'EXECUTE_____', 'FS_ACCESS____', 'FS_ACCESS()____', 'FS_ACCESS(CREATE)____', 'FS_ACCESS(CREATE__APPEND)__', 'FS_ACCESS(CREATE__READ)__', 'FS_ACCESS(CREATE__READ__WRITE)', 'FS_ACCESS(CREATE__WRITE)__', 'FS_ACCESS(CREATE__WRITE__APPEND)', 'FS_ACCESS(READ)____', 'FS_ACCESS(READ__WRITE)__', 'FS_ACCESS(WRITE)____', 'FS_PIPE_ACCESS___', 'FS_PIPE_ACCESS()___', 'FS_PIPE_ACCESS(READ)___', 'FS_PIPE_ACCESS(READ__)_', 'FS_PIPE_ACCESS(READ__WRITE)_', 'FS_PIPE_ACCESS(WRITE)___', 'NETWORK_ACCESS____', 'NETWORK_ACCESS()____', 'NETWORK_ACCESS(READ)____', 'NETWORK_ACCESS(READ__WRITE)__', 'NETWORK_ACCESS(READ__WRITE__)']


In [23]:

import numpy as np

# Get the underlying LightGBM Booster
booster = champion_model.booster_

feature_names = booster.feature_name()

print("Total model features:", len(feature_names))
print("\nLast 30 features:")
print(feature_names[-30:])

print("\nModel parameters:")
print("Number of trees:", booster.num_trees())
print("Number of classes:", booster.num_model_per_iteration())


Total model features: 470

Last 30 features:
['stop', 'stopListening', 'stopUsingNetworkFeature', 'symlink', 'sync', 'sysinfo', 'timer_create', 'timer_settime', 'toggleSoftInput', 'truncate', 'ugetrlimit', 'umask', 'uname', 'unlink', 'unregisterContentObserver', 'unregisterMediaButtonIntent', 'unregisterRemoteControlClient', 'updateAppWidgetIds', 'updateAppWidgetProvider', 'updateSelection', 'updateServiceLocation', 'utimes', 'vfork', 'vibrate', 'vibratePattern', 'wait4', 'watchRotation', 'windowGainedFocus', 'write', 'writev']

Model parameters:
Number of trees: 500
Number of classes: 5


In [24]:

import zipfile

with zipfile.ZipFile(apk_path, "r") as z:
    dex_names = [
        name for name in z.namelist()
        if name.endswith(".dex")
    ]

print("DEX files:", dex_names)

for name in dex_names:
    info = z.getinfo(name)
    print(name, "size:", round(info.file_size / 1024, 1), "KB")


DEX files: ['classes.dex']
classes.dex size: 4433.1 KB


In [25]:

import subprocess
import os
import time

smali_dir = "/content/apk_smali"
start = time.time()

result = subprocess.run(
    [
        "apktool", "d", "-f",
        "-o", smali_dir,
        apk_path
    ],
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", result.returncode)
print("Elapsed seconds:", round(time.time() - start, 2))
print(result.stderr[-1500:])

print("Smali directory exists:", os.path.isdir(smali_dir))


Return code: 0
Elapsed seconds: 18.72

Smali directory exists: True


In [26]:

import os
import re
import pandas as pd
from collections import Counter

smali_root = "/content/apk_smali"
method_counts = Counter()

for root, _, files_list in os.walk(smali_root):
    for filename in files_list:
        if not filename.endswith(".smali"):
            continue

        path = os.path.join(root, filename)

        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()

                if line.startswith("invoke-") and "->" in line:
                    match = re.search(
                        r"L([^;]+);->([^ (]+)\(([^)]*)\)(.+)",
                        line
                    )
                    if match:
                        class_name, method, args, ret = match.groups()
                        signature = (
                            f"{class_name}->{method}({args}){ret}"
                        )
                        method_counts[signature] += 1

df_methods = pd.DataFrame(
    [
        {"method_signature": method, "count": count}
        for method, count in method_counts.most_common()
    ]
)

df_methods.to_csv("/content/apk_output/dex_methods.csv", index=False)

print("Unique method references:", len(df_methods))
display(df_methods.head(20))


Unique method references: 16508


,method_signature,count
0,java/lang/Object->getClass()Ljava/lang/Class;,4831
1,java/lang/StringBuilder->append(Ljava/lang/Str...,2983
2,java/lang/Object-><init>()V,2898
3,q00->K()Ljava/lang/Object;,1859
4,x51->i(Ljava/lang/Object;Ljava/lang/Object;)Z,1533
5,java/lang/String->equals(Ljava/lang/Object;)Z,1435
6,q00->d0(Ljava/lang/Object;)V,1416
7,java/lang/Integer->valueOf(I)Ljava/lang/Integer;,1395
8,java/lang/StringBuilder->toString()Ljava/lang/...,1334
9,ha1->K(Lq00;Lmv0;Ljava/lang/Object;)V,1276


In [27]:

import os

PROJECT_DIR = "/content/drive/MyDrive/Malicious project work"

keywords = [
    "FS_ACCESS(READ)",
    "ACCESS_PERSONAL_INFO",
    "NETWORK_ACCESS",
    "champion_lightgbm_model",
]

extensions = (".py", ".ipynb", ".json", ".txt", ".csv")

print("Searching project files for feature mappings...\n")

for root, dirs, files_list in os.walk(PROJECT_DIR):
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__"}]

    for filename in files_list:
        if not filename.lower().endswith(extensions):
            continue

        path = os.path.join(root, filename)

        # Avoid reading huge datasets in full
        if os.path.getsize(path) > 20 * 1024 * 1024:
            continue

        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()

            found = [k for k in keywords if k in content]
            if found:
                print("File:", path)
                print("Matched:", found, "\n")
        except OSError:
            pass


Searching project files for feature mappings...

File: /content/drive/MyDrive/Malicious project work/feature_vectors_syscallsbinders_frequency_5_Cat.csv
Matched: ['FS_ACCESS(READ)', 'ACCESS_PERSONAL_INFO', 'NETWORK_ACCESS'] 



In [28]:
import pandas as pd
import os

csv_path = "/content/drive/MyDrive/Malicious project work/feature_vectors_syscallsbinders_frequency_5_Cat.csv"

# Read only the column names first
df_columns = pd.read_csv(csv_path, nrows=0)
dataset_features = df_columns.columns.tolist()

model_features = champion_model.booster_.feature_name()

print("Dataset columns:", len(dataset_features))
print("Model features:", len(model_features))

print("\nFirst 20 dataset columns:")
print(dataset_features[:20])

print("\nFirst 20 model features:")
print(model_features[:20])

print("\nFeatures present in model but missing from dataset:")
print([f for f in model_features if f not in dataset_features][:30])

print("\nFeatures present in dataset but missing from model:")
print([f for f in dataset_features if f not in model_features][:30])

Dataset columns: 471
Model features: 470

First 20 dataset columns:
['ACCESS_PERSONAL_INFO___', 'ALTER_PHONE_STATE___', 'ANTI_DEBUG_____', 'CREATE_FOLDER_____', 'CREATE_PROCESS`_____', 'CREATE_THREAD_____', 'DEVICE_ACCESS_____', 'EXECUTE_____', 'FS_ACCESS____', 'FS_ACCESS()____', 'FS_ACCESS(CREATE)____', 'FS_ACCESS(CREATE__APPEND)__', 'FS_ACCESS(CREATE__READ)__', 'FS_ACCESS(CREATE__READ__WRITE)', 'FS_ACCESS(CREATE__WRITE)__', 'FS_ACCESS(CREATE__WRITE__APPEND)', 'FS_ACCESS(READ)____', 'FS_ACCESS(READ__WRITE)__', 'FS_ACCESS(WRITE)____', 'FS_PIPE_ACCESS___']

First 20 model features:
['ACCESS_PERSONAL_INFO___', 'ALTER_PHONE_STATE___', 'ANTI_DEBUG_____', 'CREATE_FOLDER_____', 'CREATE_PROCESS`_____', 'CREATE_THREAD_____', 'DEVICE_ACCESS_____', 'EXECUTE_____', 'FS_ACCESS____', 'FS_ACCESS()____', 'FS_ACCESS(CREATE)____', 'FS_ACCESS(CREATE__APPEND)__', 'FS_ACCESS(CREATE__READ)__', 'FS_ACCESS(CREATE__READ__WRITE)', 'FS_ACCESS(CREATE__WRITE)__', 'FS_ACCESS(CREATE__WRITE__APPEND)', 'FS_ACCESS(REA

In [29]:
import pandas as pd

csv_path = "/content/drive/MyDrive/Malicious project work/feature_vectors_syscallsbinders_frequency_5_Cat.csv"

# Load dataset
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["Class"].value_counts().sort_index())

print("\nSample feature values:")
print(df.iloc[:5, :12].to_string())

print("\nFeature value types:")
print(df.iloc[:, :-1].dtypes.value_counts())

print("\nUnique values in first 10 features:")
for col in df.columns[:10]:
    print(col, ":", sorted(df[col].dropna().unique())[:15])

Dataset shape: (11598, 471)

Class distribution:
Class
1    1253
2    2100
3    3904
4    2546
5    1795
Name: count, dtype: int64

Sample feature values:
   ACCESS_PERSONAL_INFO___  ALTER_PHONE_STATE___  ANTI_DEBUG_____  CREATE_FOLDER_____  CREATE_PROCESS`_____  CREATE_THREAD_____  DEVICE_ACCESS_____  EXECUTE_____  FS_ACCESS____  FS_ACCESS()____  FS_ACCESS(CREATE)____  FS_ACCESS(CREATE__APPEND)__
0                        1                     0                0                   3                     0                  14                   2             0              3                0                      3                            0
1                        3                     0                0                   6                     0                  42                  91             0             32                0                     90                            0
2                        2                     0                0                   4                     0

In [30]:
import os
import re
import json
import pandas as pd
import xml.etree.ElementTree as ET
from collections import Counter

APK_PATH = apk_path
MANIFEST_PATH = "/content/apk_decoded/AndroidManifest.xml"
SMALI_DIR = "/content/apk_smali"
OUT_DIR = "/content/apk_output"

os.makedirs(OUT_DIR, exist_ok=True)

# 1. Read Android manifest
tree = ET.parse(MANIFEST_PATH)
root = tree.getroot()

ANDROID_NS = "{http://schemas.android.com/apk/res/android}"

permissions = sorted({
    elem.attrib.get(ANDROID_NS + "name", "")
    for elem in root.findall("uses-permission")
    if elem.attrib.get(ANDROID_NS + "name")
})

application = root.find("application")
activities = root.findall(".//activity")
services = root.findall(".//service")
receivers = root.findall(".//receiver")
providers = root.findall(".//provider")

package_name = root.attrib.get("package", "Unknown")

# 2. Analyze existing Smali files without re-decoding
method_refs = Counter()
api_indicators = {
    "dynamic_code_loading": [
        "DexClassLoader", "PathClassLoader", "InMemoryDexClassLoader"
    ],
    "command_execution": [
        "java/lang/Runtime;->exec",
        "Ljava/lang/ProcessBuilder;->start"
    ],
    "reflection": [
        "Ljava/lang/reflect/Method;->invoke",
        "Ljava/lang/Class;->forName"
    ],
    "networking": [
        "Ljava/net/Socket;",
        "Ljava/net/URL;->openConnection",
        "Lokhttp3/",
        "Ljavax/net/ssl/"
    ],
    "crypto": [
        "Ljavax/crypto/",
        "Ljava/security/MessageDigest;"
    ],
    "native_code": [
        "Ljava/lang/System;->loadLibrary",
        "Ljava/lang/Runtime;->loadLibrary"
    ],
}

indicator_hits = Counter()
smali_files = 0

for base, _, files in os.walk(SMALI_DIR):
    for filename in files:
        if not filename.endswith(".smali"):
            continue

        smali_files += 1
        path = os.path.join(base, filename)

        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    line = line.strip()

                    if line.startswith("invoke-"):
                        match = re.search(
                            r"(L[^;]+;->[^\s(]+(?:\([^)]*\).*)?)",
                            line
                        )
                        if match:
                            method_refs[match.group(1)] += 1

                    for category, patterns in api_indicators.items():
                        if any(p in line for p in patterns):
                            indicator_hits[category] += 1

        except OSError:
            pass

# 3. Build structured report
report = {
    "apk_file": os.path.basename(APK_PATH),
    "package_name": package_name,
    "static_analysis_only": True,
    "model_prediction": None,
    "model_prediction_note": (
        "Not generated: extracted features have not been validated "
        "against the LightGBM training feature-generation pipeline."
    ),
    "manifest": {
        "permissions": permissions,
        "permission_count": len(permissions),
        "activity_count": len(activities),
        "service_count": len(services),
        "receiver_count": len(receivers),
        "provider_count": len(providers),
        "application_debuggable": (
            application.attrib.get(ANDROID_NS + "debuggable")
            if application is not None else None
        ),
    },
    "dex_analysis": {
        "smali_file_count": smali_files,
        "unique_method_reference_count": len(method_refs),
        "total_method_reference_count": sum(method_refs.values()),
        "api_indicator_line_hits": dict(indicator_hits),
        "top_method_references": [
            {"method": method, "count": count}
            for method, count in method_refs.most_common(100)
        ],
    },
}

# Save JSON report
json_path = os.path.join(OUT_DIR, "apk_analysis_report.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

# Save permissions
pd.DataFrame({"permission": permissions}).to_csv(
    os.path.join(OUT_DIR, "apk_permissions.csv"), index=False
)

# Save method reference counts
pd.DataFrame(
    method_refs.most_common(),
    columns=["method_reference", "count"]
).to_csv(
    os.path.join(OUT_DIR, "dex_methods.csv"), index=False
)

# Save API indicator counts
pd.DataFrame(
    [{"indicator": k, "line_hits": v}
     for k, v in indicator_hits.items()]
).to_csv(
    os.path.join(OUT_DIR, "api_indicators.csv"), index=False
)

print("APK static analysis complete.")
print("Package:", package_name)
print("Permissions:", len(permissions))
print("Smali files:", smali_files)
print("Unique method references:", len(method_refs))
print("API indicator hits:", dict(indicator_hits))
print("\nReport saved:", json_path)
print("Output directory:", OUT_DIR)

APK static analysis complete.
Package: com.iris.gallery
Permissions: 14
Smali files: 4415
Unique method references: 16508
API indicator hits: {'networking': 355, 'reflection': 97, 'crypto': 11, 'native_code': 2}

Report saved: /content/apk_output/apk_analysis_report.json
Output directory: /content/apk_output


In [31]:
import json
import os

report_path = "/content/apk_output/apk_analysis_report.json"

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

permissions = report["manifest"]["permissions"]
indicators = report["dex_analysis"]["api_indicator_line_hits"]

# These are review flags, not a malware verdict.
review_flags = []

if any("REQUEST_INSTALL_PACKAGES" in p for p in permissions):
    review_flags.append("Requests permission related to installing packages")

if any("SYSTEM_ALERT_WINDOW" in p for p in permissions):
    review_flags.append("Requests overlay-window capability")

if any("MANAGE_EXTERNAL_STORAGE" in p for p in permissions):
    review_flags.append("Requests broad external-storage access")

if indicators.get("dynamic_code_loading", 0) > 0:
    review_flags.append("Dynamic code-loading patterns found")

if indicators.get("command_execution", 0) > 0:
    review_flags.append("Command-execution patterns found")

if indicators.get("reflection", 0) > 0:
    review_flags.append("Reflection patterns found; review context")

if indicators.get("networking", 0) > 0:
    review_flags.append("Networking patterns found; inspect destinations and behavior")

report["review_flags"] = review_flags
report["assessment"] = {
    "status": "Requires further investigation",
    "reason": (
        "Static indicators alone cannot establish whether an APK is malicious. "
        "No validated LightGBM prediction was produced."
    )
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Review flags:")
for flag in review_flags:
    print("-", flag)

if not review_flags:
    print("- No configured review flags triggered.")

print("\nUpdated report:", report_path)

Review flags:
- Requests broad external-storage access
- Reflection patterns found; review context
- Networking patterns found; inspect destinations and behavior

Updated report: /content/apk_output/apk_analysis_report.json
